In [1]:
from source.toomanycells import toomanycells as tmc
import pandas as pd
import numpy as np
import json
import os

> 7/7 packages are installed:
pandas package is Installed (v2.3.3)
numpy package is Installed (v2.2.6)
scipy package is Installed (v1.14.1)
tqdm package is Installed (v4.67.0)
pandarallel package is Installed (v1.6.5)
sqlalchemy package is Installed (v2.0.38)
pymysql package is Installed (v1.1.1)


In [2]:
import pandas as pd
import json
import sys

# Increase recursion limit for the deep tree (approx 41 levels)
sys.setrecursionlimit(20000)

# 1. Load the data
clusters_df = pd.read_csv('tmc_output\\covid_vaccine_new-subject7\\cells_clusters_info.csv')
labels_df = pd.read_csv('tmc_input\\covid_vaccine_new-subject7\\demo_label.csv')

# Use the first column (barcodes) and rename for clarity
clusters_df.rename(columns={clusters_df.columns[0]: 'item'}, inplace=True)

# 2. Map items to their specific labels
item_label_map = dict(zip(labels_df['item'], labels_df['label']))

# 3. Build adjacency maps for the sp tree
cluster_to_children = {} # parent cluster -> set of child clusters
cluster_to_items = {}    # leaf cluster -> list of (barcode, label)

for _, row in clusters_df.iterrows():
    path = str(row['sp_path']).split('/') # e.g., ["25523", ..., "0"]
    item = row['item']
    label = item_label_map.get(item)
    
    # Map the item to its most specific sp cluster (the first in the path)
    leaf_cluster = path[0]
    if leaf_cluster not in cluster_to_items:
        cluster_to_items[leaf_cluster] = []
    cluster_to_items[leaf_cluster].append((item, label))
    
    # Build hierarchy: path[i] is a child of path[i+1]
    for i in range(len(path) - 1):
        child = path[i]
        parent = path[i+1]
        if parent not in cluster_to_children:
            cluster_to_children[parent] = set()
        cluster_to_children[parent].add(child)

# 4. Recursive function to build the [metadata, children] tuple format
def build_clumpiness_node(node_id):
    # The first element is the metadata object
    metadata = {
        "nodeID": str(node_id),
        "nodeLabels": [] # Internal nodes usually have empty labels
    }
    
    # The second element is the list of children nodes
    children = []
    
    # Add child clusters
    if node_id in cluster_to_children:
        for child_id in cluster_to_children[node_id]:
            children.append(build_clumpiness_node(child_id))
            
    # Add individual cell items as leaf nodes
    if node_id in cluster_to_items:
        for item_id, label in cluster_to_items[node_id]:
            leaf_metadata = {
                "nodeID": str(item_id),
                "nodeLabels": [str(label)] if label else []
            }
            # A leaf is a 2-element list with an empty children list: [meta, []]
            children.append([leaf_metadata, []])
            
    return [metadata, children]

# 5. Generate tree from root '0' and save
final_tree = build_clumpiness_node('0')

with open('find_clumpiness_input.json', 'w') as f:
    json.dump(final_tree, f)

In [11]:
import json
path_tree = "tmc_output\\covid_vaccine_new-subject7\\cluster_tree.json"

# Open and load the JSON file
with open(path_tree, 'r',) as file:
    data = json.load(file)

# Open a file in write mode ('w')
with open("json_dump.json", 'w') as f:
    json.dump(data, f)

In [6]:
data[0]

{'_item': None, '_significance': None, '_distance': 0.3388460032304374}

In [ ]:
data

KeyboardInterrupt: 